# 02 — Preparar perfiles de voz con XTTS-v2

> **Entrega final · módulo de personalización de voz**

Este notebook construye un perfil reutilizable para el asistente de compras por voz a partir de una grabación autorizada. El proceso **no entrena un modelo nuevo**: XTTS-v2 extrae las representaciones de la voz y las almacena en un archivo `.pt`, que luego puede emplearse para sintetizar nuevas frases.

### Resultado esperado

Para una persona identificada como `winz`, el audio `data/voice_samples/winz.wav` genera el artefacto `data/voice_profiles/winz.pt`. El mismo procedimiento se repite cambiando `VOICE_ID` para crear perfiles adicionales.

### Flujo de trabajo

1. Verificar el dispositivo disponible y cargar XTTS-v2.
2. Seleccionar una grabación autorizada y su identificador.
3. Calcular y guardar las representaciones de voz.
4. Validar el perfil mediante una frase nueva.
5. Integrar el archivo generado en el asistente de compras.

## 1. Preparación del entorno y comprobación de CUDA

La celda siguiente detecta automáticamente si hay una GPU CUDA disponible. El notebook también puede ejecutarse en CPU, aunque la carga del modelo y la síntesis serán más lentas. Para reproducir la entrega, utiliza el entorno del proyecto y las dependencias declaradas en `requirements.txt`.

In [1]:
import os
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)
print('CUDA disponible:', torch.cuda.is_available())


Dispositivo: cuda
CUDA disponible: True


## 2. Cargar el modelo XTTS-v2

Se carga el checkpoint multilingüe oficial de XTTS-v2. La primera ejecución puede descargar el modelo y tardar varios minutos; las siguientes ejecuciones reutilizan los archivos locales disponibles. La configuración de aceptación de los términos de Coqui se aplica antes de inicializar el modelo.

PyTorch reciente restringe los objetos que puede reconstruir desde checkpoints. La celda registra explícitamente la configuración de XTTS que requiere el checkpoint oficial.

In [2]:
import torch
from TTS.tts.configs.xtts_config import XttsConfig
torch.serialization.add_safe_globals([XttsConfig])
os.environ["COQUI_TOS_AGREED"] = "1"

# El checkpoint oficial de XTTS es confiable y requiere carga pickle completa.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

from TTS.api import TTS

tts = TTS(
    "tts_models/multilingual/multi-dataset/xtts_v2",
    gpu=torch.cuda.is_available(),
)
model = tts.synthesizer.tts_model

print("XTTS cargado")

/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


XTTS cargado


## 3. Seleccionar la identidad y la grabación de referencia

`VOICE_ID` identifica de forma única a la persona y debe coincidir con el nombre del archivo WAV ubicado en `data/voice_samples/`. Por ejemplo, `VOICE_ID = "winz"` busca `data/voice_samples/winz.wav`.

Para obtener un perfil estable, utiliza una grabación de entre 10 y 30 segundos, en un ambiente silencioso, con voz natural, volumen constante y sin música ni conversaciones de fondo. Cada perfil debe corresponder a una sola persona y contar con su autorización explícita.

In [3]:
# Cambia este identificador para crear otro perfil (ej.: "winz", "persona_02").
VOICE_ID = "winz"

from pathlib import Path

# Encontrar la raíz aunque Jupyter se haya iniciado desde notebooks/.
PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "data" / "voice_samples").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró data/voice_samples. Inicia Jupyter desde la raíz del proyecto."
    )

VOICE_SAMPLES_DIR = PROJECT_DIR / "data" / "voice_samples"
VOICE_PROFILES_DIR = PROJECT_DIR / "data" / "voice_profiles"
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
voice_sample_path = VOICE_SAMPLES_DIR / f"{VOICE_ID}.wav"
if not voice_sample_path.is_file():
    disponibles = sorted(p.name for p in VOICE_SAMPLES_DIR.glob("*.wav"))
    raise FileNotFoundError(
        f"No existe la grabación: {voice_sample_path.resolve()}\n"
        f"Archivos WAV disponibles: {disponibles or 'ninguno'}"
    )
print("Raíz del proyecto:", PROJECT_DIR)
print("Persona seleccionada:", VOICE_ID)
print("Audio seleccionado:", voice_sample_path.resolve())

Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz
Persona seleccionada: winz
Audio seleccionado: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/voice_samples/winz.wav


## 4. Extraer las representaciones de voz

XTTS calcula dos representaciones internas a partir del audio de referencia: `gpt_cond_latent` y `speaker_embedding`. Guardarlas evita repetir este cálculo en cada sesión. El modelo XTTS-v2 sigue siendo necesario para sintetizar el audio final.

In [4]:
import torch
import torchaudio

print("PyTorch:", torch.__version__)
print("CUDA de PyTorch:", torch.version.cuda)
print("TorchAudio:", torchaudio.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Capacidad GPU:", torch.cuda.get_device_capability(0))
print("Arquitecturas de PyTorch:", torch.cuda.get_arch_list())

torch.arange(3, device="cuda")
print("Prueba básica de CUDA: OK")

PyTorch: 2.7.1+cu128
CUDA de PyTorch: 12.8
TorchAudio: 2.7.1+cu128
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
Capacidad GPU: (12, 0)
Arquitecturas de PyTorch: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120', 'compute_120']
Prueba básica de CUDA: OK


In [5]:
with torch.inference_mode():
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=[str(voice_sample_path)],
        gpt_cond_len=6,
        max_ref_length=30,
        sound_norm_refs=False
    )
print("Perfil calculado:", VOICE_ID)
print("Dimensiones:", tuple(gpt_cond_latent.shape), tuple(speaker_embedding.shape))

Perfil calculado: winz
Dimensiones: (1, 32, 1024) (1, 512, 1)


## 5. Persistir el artefacto generado

El perfil se guarda como `data/voice_profiles/<VOICE_ID>.pt` e incluye el identificador, las representaciones calculadas, el modelo utilizado y el idioma de referencia. Este archivo es el artefacto que se reutilizará en el asistente.

Verifica que la ruta se haya creado correctamente y que el archivo no se incorpore a un repositorio público: tanto la grabación como el perfil contienen información biométrica potencialmente identificable.

In [6]:
VOICE_PROFILES_DIR.mkdir(parents=True, exist_ok=True)
profile_path = VOICE_PROFILES_DIR / f"{VOICE_ID}.pt"
torch.save({
    "voice_id": VOICE_ID,
    "gpt_cond_latent": gpt_cond_latent.detach().cpu(),
    "speaker_embedding": speaker_embedding.detach().cpu(),
    "model": "XTTS-v2",
    "language": "es"
}, profile_path)
print("Perfil guardado:", profile_path)

Perfil guardado: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/voice_profiles/winz.pt


## 6. Validar la síntesis

La prueba carga el perfil recién guardado y sintetiza una frase que no pertenece a la grabación de referencia. La salida permite comprobar que el archivo `.pt` es legible y que XTTS puede reutilizar la identidad vocal.

### Criterio de aceptación

La ejecución se considera correcta cuando se obtiene el audio sintetizado sin errores, el perfil existe en `data/voice_profiles/` y la voz resultante es razonablemente consistente con la grabación autorizada. La calidad puede variar según el ruido, la duración y la expresividad del audio de referencia.

In [10]:
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = "Hola, donde esta mi tonchito y lukitas"
with torch.inference_mode():
    result = model.inference(
        text=text, language="es",
        gpt_cond_latent=profile["gpt_cond_latent"].to(DEVICE),
        speaker_embedding=profile["speaker_embedding"].to(DEVICE),
        enable_text_splitting=True
    )
from IPython.display import Audio, display
display(Audio(result["wav"], rate=24000))

## 7. Integrar el perfil en el asistente de compras

Copia el archivo `.pt` generado a la carpeta de perfiles de Google Drive configurada en `Lab4_Asistente_Compras_por_Voz_MVP.ipynb`. Allí, ajusta `VOICE_ID` para seleccionar el perfil que utilizará XTTS; por ejemplo, `winz` corresponde a `winz.pt` y `persona_02` a `persona_02.pt`.

### Checklist de entrega

- El audio de referencia está en `data/voice_samples/` y corresponde al identificador elegido.
- El perfil `.pt` fue generado en `data/voice_profiles/`.
- La frase de validación se sintetizó correctamente.
- El perfil fue copiado a la ubicación usada por el notebook MVP.
- `VOICE_ID` está configurado con el perfil que se desea utilizar.

**Privacidad y uso responsable.** Una grabación y su perfil pueden reproducir una voz identificable. Conserva ambos archivos en un espacio privado, limita el acceso y utiliza únicamente voces propias o aquellas para las que exista autorización explícita.